# MHA
不多说，直接开撕


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
class MultiHeadAtten(nn.Module):
    def __init__(self,num_heads,dim):
        super().__init__()
        
        
        self.num_heads = num_heads
        self.dim = dim
        assert self.dim % num_heads == 0
        self.head_dim = self.dim // self.num_heads
        self.W_Q = nn.Linear(self.dim,self.dim)
        self.W_K = nn.Linear(self.dim,self.dim)
        self.W_V = nn.Linear(self.dim,self.dim)
        self.W_O = nn.Linear(self.dim,self.dim)
    def forward(self,x):
        B = x.shape[0]
        T = x.shape[1]
        # 线性层QKV
        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)

        # 切分多头
        Q_h = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K_h = K.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V_h = V.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # 计算注意力分数
        scores = torch.matmul(Q_h,K_h.transpose(-2,-1)) / math.sqrt(self.head_dim)
        attn = F.softmax(scores,-1)
        output =torch.matmul(attn,V_h)

        # 拼接
        concat = output.transpose(1, 2).contiguous().view(B, T, self.dim)
        output = self.W_O(concat)
        return output
 


        


In [3]:
x = torch.randn(2,4,8)
model = MultiHeadAtten(num_heads=2,dim=8)
_ = model(x)
print(model)
print(x.shape)
print(model(x).shape)


NotImplementedError: Module [MultiHeadAtten] is missing the required "forward" function